In [ ]:
import pandas as pd 
import json 
import plotly.graph_objects as go 
from common import atp_to_comuni, comuni_to_id
from pathlib import Path 
from data_preparation.v2.utils.utils import download_s3_folder, get_dataframe, get_s3, get_json_s3
from plotly.subplots import make_subplots
import numpy 
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
download = False

DOWNLOAD_DIR = './downloaded_files'
RIFIUTI_PATH = Path(DOWNLOAD_DIR) / "Rifiuti"
DEPURATORE_PATH = Path(DOWNLOAD_DIR) / "Depurazione"

if download: 
    download_s3_folder("Depurazione", local_dir=DOWNLOAD_DIR + "/Depurazione")
    download_s3_folder("Rifiuti", local_dir=DOWNLOAD_DIR + "/Rifiuti")

In [ ]:
vodafone_presenze = get_dataframe("vodafone_attendences")
geojson_comuni_json_data = json.load(get_s3("TRENTINO-comuni_Vodafone_2023.geojson"))
json_vodafone = get_json_s3("mapping_ids/mapping_comuni_into_vodafone_Trento.json") 
json_apt = get_json_s3("mapping_ids/map_comuni_into_apt.json")
json_comuni = get_json_s3("mapping_ids/mapping_comuni_ISTAT.json")

In [ ]:
location_map = {
        feature["properties"]["id"]: feature["properties"]["name"].upper()
        for feature in geojson_comuni_json_data["features"]
    }

## Filtraggio e assegnazione dei comuni
vodafone_presenze["comune"] = vodafone_presenze["locId"].map(
        location_map
    )
vodafone_presenze["ID_COMUNE"] = vodafone_presenze["comune"].map(
        json_vodafone
)
vodafone_presenze = vodafone_presenze[
        (vodafone_presenze["locType"] == "TN_MKT_AL_3")]  # filtriamo sui soli comuni 

mask = vodafone_presenze["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])

vodafone_presenze.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA"
vodafone_presenze.loc[mask, "ID_COMUNE"] = [[22250]] * mask.sum()
vodafone_presenze

# Analisi dei dati di rifiuti 
#### I dati dei rifiuti contengono, per ogni comunita', i quantitativi mensili di rifiuto smaltiti, divisi per tipologia, con il rispettivao totale. I dati fanno riferimetno all'anno 2024

In [ ]:
rifiuti_DF = pd.read_excel(RIFIUTI_PATH / "dati_ORSO_2022-2024.xlsx")
rifiuti_DF

In [ ]:
## MAPPING APT <> COMUNITA' 
mapping_1to1 = {
    # Comuni della Val di Fassa -> APT Val di Fassa
    "Campitello di Fassa": "Val di Fassa",
    "Canazei": "Val di Fassa",
    "Mazzin": "Val di Fassa",
    "Moena": "Val di Fassa",
    "Sèn Jan di Fassa-Sèn Jan": "Val di Fassa",
    "Soraga di Fassa": "Val di Fassa",
    
    # Rovereto e Vallagarina -> APT Rovereto, Vallagarina e Monte Baldo
    "Rovereto": "Rovereto, Vallagarina e Monte Baldo",
    "Comunità Vallagarina": "Rovereto, Vallagarina e Monte Baldo",
    
    # Capoluogo e Val d'Adige -> APT Trento, Monte Bondone e Altopiano di Pinè
    "Trento": "Trento, Monte Bondone e Altopiano di Pinè",
    "Comunità Val d’Adige": "Trento, Monte Bondone e Altopiano di Pinè",
    
    # Comunità standard a corrispondenza diretta
    "Comunità Primiero": "San Martino di Castrozza, Primiero e Vanoi",
    "Comunità Val di Non": "Val di Non",
    "Comunità Val di Sole": "Val di Sole",
    "Comunità Val di Fiemme": "Val di Fiemme e Val di Cembra",
    "Comunità Alto Garda e Ledro": "Garda trentino, Valle di Ledro, Terme di Comano e Valle dei Laghi",
    "Comunità Valsugana e Tesino": "Valsugana, Tesino e Valle dei Mocheni",
    
    # Scelte di prevalenza per le comunità-APT:
    "Comunità Giudicarie": "Madonna di Campiglio, Pinzolo, Val Rendena, Giudicarie centrali e Valle del Chiese",
    "Comunità Alta Valsugana": "Valsugana, Tesino e Valle dei Mocheni"
}
rifiuti_DF['APT'] = rifiuti_DF['Comune'].map(mapping_1to1)
assert (set(rifiuti_DF['Comune'].unique()) - set(mapping_1to1.keys())) == set(), "Alcuni comuni non sono stati mappati correttamente."

In [ ]:
rifiuti_DF = rifiuti_DF.fillna(0)
rifiuti_DF

In [ ]:
# check
month_cols = [
    'tot(kg)_gen',
    'tot(kg)_feb',
    'tot(kg)_mar',
    'tot(kg)_apr',
    'tot(kg)_mag',
    'tot(kg)_giu',
    'tot(kg)_lug',
    'tot(kg)_ago',
    'tot(kg)_set',
    'tot(kg)_ott',
    'tot(kg)_nov',
    'tot(kg)_dic',
]

# Mappatura dei nomi dei mesi
mappa_mesi = {
    'tot(kg)_gen': 1,
    'tot(kg)_feb': 2,
    'tot(kg)_mar': 3,
    'tot(kg)_apr': 4,
    'tot(kg)_mag': 5,
    'tot(kg)_giu': 6,
    'tot(kg)_lug': 7,
    'tot(kg)_ago': 8,
    'tot(kg)_set': 9,
    'tot(kg)_ott': 10,
    'tot(kg)_nov': 11,
    'tot(kg)_dic': 12,
}


print("Valori sospetti:")
rifiuti_DF[~numpy.isclose(
    rifiuti_DF['totSommaMens(kg)'], rifiuti_DF[month_cols].sum(axis=1)
)]

In [ ]:
# Trasformazione del dataframe
df_melt_rifiuti = rifiuti_DF.melt(
    id_vars=[
        'Regione',
        'Sigla',
        'Comune',
        'Anno',
        'Macro',
        'RifiutoCompleto',
        'CER',
        'Provincia',
        'P',
        'APT'
    ],
    value_vars=month_cols,
    var_name='Mese',
    value_name='Kg',
)

df_melt_rifiuti['Mese'] = df_melt_rifiuti['Mese'].map(mappa_mesi)
vodafone_presenze

In [ ]:
vodafone_presenze = vodafone_presenze.dropna(subset=['ID_COMUNE'])  # droppiamo le righe senza ID_COMUNE

vodafone_presenze['date'] = pd.to_datetime(vodafone_presenze['date'])
vodafone_presenze['mese'] = vodafone_presenze['date'].dt.to_period("M").dt.to_timestamp()
# Divido il valore in parti uguali per preparare all'explode 
vodafone_presenze['n_elementi'] = vodafone_presenze['ID_COMUNE'].apply(
    lambda x: len(x) if isinstance(x, list) else 1
)
vodafone_presenze['value'] = (
    vodafone_presenze['value'] / vodafone_presenze['n_elementi']
)

vodafone_presenze = vodafone_presenze.explode('ID_COMUNE')
vodafone_presenze['ID_COMUNE'] = vodafone_presenze['ID_COMUNE'].astype(str).str.zfill(6)

vodafone_presenze['comune'] = vodafone_presenze['ID_COMUNE'].map(comuni_to_id(json_comuni))  # mappiamo ogni comune correttamnete, ora che il DF e' esploso 
vodafone_presenze['APT'] = vodafone_presenze['ID_COMUNE'].map(atp_to_comuni(json_apt))   # mappiamo ogni comune all'APT corrispondente

In [ ]:
vodafone_monthly_apt = vodafone_presenze.groupby(["mese", "APT", "userProfile"], as_index=False).agg({'value' : 'sum'})  # sommiamo i valori per mese, APT e userProfile
vodafone_monthly_apt

In [ ]:
df_melt_rifiuti['mese_str'] = pd.to_datetime(df_melt_rifiuti['Anno'].astype(str) + '-' + df_melt_rifiuti['Mese'].astype(str).str.zfill(2))
df_melt_rifiuti_total = df_melt_rifiuti.groupby(["mese_str", "APT"]).agg({'Kg': 'sum'}).reset_index()   # sommiamo i valori dei rifiuti per mese e APT
df_melt_rifiuti_total

## Confronto distribuzione MENSILE+APT delle PRESENZE VODAFONE e delle DITRIBUZIONI DEI RIFIUTI: trend e correlazioni 

In [ ]:
userprofile = ['TOURIST', 'INHABITANT']  # I profili da confrontare 
apts = vodafone_monthly_apt['APT'].unique()

vodafone_monthly_apt['anno'] = vodafone_monthly_apt['mese'].dt.year
vodafone_monthly_apt['mese_num'] = vodafone_monthly_apt['mese'].dt.month

df_melt_rifiuti_total['anno'] = df_melt_rifiuti_total['mese_str'].dt.year
df_melt_rifiuti_total['mese_num'] = df_melt_rifiuti_total['mese_str'].dt.month

# Filtra sui profili scelti e somma i valori per APT, Anno e Mese
df_presenze_prep = (
    vodafone_monthly_apt[vodafone_monthly_apt['userProfile'].isin(userprofile)]
    .groupby(['APT', 'anno', 'mese_num'], as_index=False)['value']
    .sum()
    .sort_values('mese_num')
)

df_rifiuti_prep = df_melt_rifiuti_total.sort_values('mese_num')

for aa in apts:
    print(f"APT: {aa}")
    presenze_apt = df_presenze_prep[df_presenze_prep['APT'] == aa]
    rifiuti_apt = df_rifiuti_prep[df_rifiuti_prep['APT'] == aa]

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    for anno, dati in presenze_apt.groupby('anno'):
        fig.add_trace(
            go.Scatter(
                x=dati['mese_num'],
                y=dati['value'],
                mode='lines+markers',
                name=f'Presenze {anno}',
            ),
            secondary_y=False,
        )

    for anno, dati in rifiuti_apt.groupby('anno'):
        fig.add_trace(
            go.Scatter(
                x=dati['mese_num'],
                y=dati['Kg'],
                mode='lines+markers',
                name=f'Rifiuti {anno}',
                line=dict(dash='dot'),
            ),
            secondary_y=True,
        )
    mean_df_presenze = presenze_apt.groupby('mese_num').agg({"value": "mean"})
    fig.add_trace(
        go.Scatter(
            x = mean_df_presenze.index,
            y = mean_df_presenze.value,
            mode='lines+markers',
                name=f'Presenze, apt, media annuale',
                line=dict(dash='dot', color = "orange"),
            )
        )

    fig.update_layout(
        title=f'<b>{aa}</b> - Trend annuale (Profili: {", ".join(userprofile)})',
        xaxis=dict(
            tickmode='array', tickvals=list(range(1, 13)), ticktext=[
                'Gen',
                'Feb',
                'Mar',
                'Apr',
                'Mag',
                'Giu',
                'Lug',
                'Ago',
                'Set',
                'Ott',
                'Nov',
                'Dic',
            ]

        ),
        template='plotly_white',
    )
    fig.update_yaxes(title_text='Presenze Totali', secondary_y=False)
    fig.update_yaxes(title_text='Rifiuti (Kg)', secondary_y=True)

    fig.show()

In [ ]:
presenze_medie = df_presenze_prep.groupby(['APT', 'mese_num'], as_index=False)[
    'value'
].mean()

rifiuti_2024 = (
    df_rifiuti_prep
    .groupby(['APT', 'mese_num'], as_index=False)['Kg']
    .sum()
)

df_corr = pd.merge(
    presenze_medie, 
    rifiuti_2024, 
    on=['APT', 'mese_num'])

# Calcolo correlazione per ogni APT
correlazioni_apt = (
    df_corr.groupby('APT')[['value']]
    .corrwith(df_corr['Kg'])
    .reset_index()
    .rename(columns={'value': 'Correlazione'})
).sort_values('Correlazione', ascending=False)
# corr_per_apt = (df_corr.groupby('APT').apply(lambda g: g['value'].corr(g['Kg'])).reset_index(name='correlazione').sort_values('correlazione', ascending=False))
print('--- CORRELAZIONE PER APT ---')
print(correlazioni_apt)

In [ ]:
## NB: nella versione precedente abbiamo supposto un mapping 1:1 tra APT e Comunità, ma in realtà alcune Comunità sono presenti in più APT e viceversa.
## QUESTO SAREBBE IL MAPPING CORRETTO (non 1:1)

# mapping = {
#     "Val di Non": [
#         "Comunità Val di Non"
#     ],
#     "San Martino di Castrozza, Primiero e Vanoi": [

#         "Comunità Primiero"
#     ],
#     "Rovereto, Vallagarina e Monte Baldo": [
#         "Rovereto",
#         "Comunità Vallagarina"
#     ],
#     "Val di Fassa": [
#         "Campitello di Fassa",
#         "Canazei",
#         "Mazzin",
#         "Moena",
#         "Sèn Jan di Fassa-Sèn Jan",
#         "Soraga di Fassa"
#     ],
#     "Altopiano della Paganella, Piana della Rotaliana e San Lorenzo Dorsino": [
#         "Comunità Giudicarie"  # Per via del comune di San Lorenzo Dorsino
#     ],
#     "Madonna di Campiglio, Pinzolo, Val Rendena, Giudicarie centrali e Valle del Chiese": [
#         "Comunità Giudicarie"
#     ],
#     "Val di Fiemme e Val di Cembra": [
#         "Comunità Val di Fiemme"
#     ],
#     "Valsugana, Tesino e Valle dei Mocheni": [
#         "Comunità Valsugana e Tesino",
#         "Comunità Alta Valsugana"  # Pergine, Levico, Caldonazzo, Mòcheni
#     ],
#     "Trento, Monte Bondone e Altopiano di Pinè": [
#         "Trento",
#         "Comunità Val d’Adige",
#         "Comunità Alta Valsugana"  # Per l'Altopiano di Piné
#     ],
#     "Garda trentino, Valle di Ledro, Terme di Comano e Valle dei Laghi": [
#         "Comunità Alto Garda e Ledro",
#         "Comunità Giudicarie"  # Per l'area delle Terme di Comano
#     ],
#     "Altipiani cimbri e Vigolana": [
#         "Comunità Alta Valsugana"  # Per l'Altopiano della Vigolana
#     ],
#     "Val di Sole": [
#         "Comunità Val di Sole"
#     ]
# }

## Note d'attenzione: 
- Le aree APT e Comunita' hanno degli overlap. Per questo, sono state fatte delle scelte di prevalenza 
- Di comunità di valle ufficiali, mancano almeno: Magnifica Comunità degli Altipiani Cimbri, Comunità Rotaliana-Königsberg, Comunità della Paganella,
Comunità della Valle dei Laghi
- dal 1° gennaio 2023 i comuni di Albiano, Bedollo, Baselga di Piné e Fornace sono stati spostati dall'ambito APT "Val di Fiemme, Altopiano di Pinè e Val di Cembra" a "Trento, Monte Bondone e Altopiano di Piné"
- come si nota anche dai grafici, Altopiano della Paganella, Piana della Rotaliana e San Lorenzo Dorsino e Altipiani cimbri e Vigolana rimangono senza una associazione diretta 

# Analisi dei dati di depurazione 

In [ ]:
# Mapping impianto di depurazione -> comune (approssimato, 1 impianto = 1 comune)

impianto_to_comune = {
    "ALA": "ALA",
    "ALBIANO": "ALBIANO",
    "ANDALO": "ANDALO",
    "ARCO": "ARCO",
    "AVIO": "AVIO",
    "BASELGA DI PINE": "BASELGA DI PINE",
    "CAMPODENNO": "CAMPODENNO",
    "CANAL SAN BOVO": "CANAL SAN BOVO",
    "CASTELLO TESINO": "CASTELLO TESINO",
    "CAVARENO": "CAVARENO",
    "CLES": "CLES",
    "DRENA": "DRENA",
    "FAI DELLA PAGANELLA": "FAI DELLA PAGANELLA",
    "FOLGARIA": "FOLGARIA",
    "GIUSTINO": "GIUSTINO",
    "GRIGNO": "GRIGNO",
    "IMER": "IMER",
    "LAVARONE": "LAVARONE",
    "LAVIS": "LAVIS",
    "LEVICO": "LEVICO TERME",
    "MALE": "MALE",
    "MEZZANA": "MEZZANA",
    "MEZZOCORONA": "MEZZOCORONA",
    "MOLVENO": "MOLVENO",
    "MORI": "MORI",
    "ROVERETO": "ROVERETO",
    "SOVER": "SOVER",
    "STENICO": "STENICO",
    "STORO": "STORO",
    "TERRAGNOLO": "TERRAGNOLO",
    "TESERO": "TESERO",
    "VALLARSA": "VALLARSA",

    # --- Match nome ISTAT leggermente diverso
    "CAMPITELLO": "CAMPITELLO DI FASSA",
    "MOENA": "MOENA",
    "MADONNA DI CAMPIGLIO": "PINZOLO",

    # --- Impianti = frazione di un comune fuso (verificati) ---
    "BANCO": "SANZENO",                           
    "CALAVINO": "MADRUZZO",                       
    "CASTELLO DI FIEMME": "CASTELLO-MOLINA DI FIEMME",
    "CHIZZOLA": "ALA",                            
    "DIMARO": "DIMARO FOLGARIDA",                 
    "DORSINO": "SAN LORENZO DORSINO",             
    "DRO PIETRAMURATA": "DRO",                    
    "FAVER": "ALTAVALLE",                         
    "FONDO": "BORGO D'ANAUNIA",
    "MOLINA DI FIEMME": "CASTELLO-MOLINA DI FIEMME",
    "MOLINA DI LEDRO": "LEDRO",                  
    "PIEVE DI BONO": "PIEVE DI BONO-PREZZO",
    "PIEVE DI LEDRO": "LEDRO",
    "POZZA DI FASSA": "SAN GIOVANNI DI FASSA", 
    "RAGOLI": "TRE VILLE",                      
    "SAN MARTINO DI CASTROZZA": "PRIMIERO SAN MARTINO DI CASTROZZA",
    "SANTA MASSENZA": "VALLELAGHI",               
    "SPIAZZO RENDENA": "SPIAZZO",                
    "TAIO": "PREDAIA",                           
    "VILLA AGNEDO": "CASTEL IVANO",
    "CARBONARE": "FOLGARIA",
    "MALGA LAGHETTO": "LAVARONE",
    "RIVA ARENA": "RIVA DEL GARDA",
    "RIVA SAN NICOLO": "RIVA DEL GARDA",
    "PIEVE CINTE TESINO": "PIEVE TESINO",

    # --- Passi montani / impianti turistici in quota (comune amministrativo) ---
    "PASSO TONALE": "VERMIGLIO", 
    "PASSO ROLLE": "PRIMIERO SAN MARTINO DI CASTROZZA",
    "PASSO LAVAZE": "VILLE DI FIEMME",

    # --- Città con più impianti -> stesso comune ---
    "TRENTO NORD": "TRENTO",
    "TRENTO SUD": "TRENTO",
}

In [ ]:
folder_path = DEPURATORE_PATH / "Elaborati"

dict_depuratori = {}
all_dfs = []
loading = False 
save = False

if loading : 
    for file_path in folder_path.glob("*.ods"):
        if file_path.name in ["REPORT.ods", "STATS.ods"]:
            continue

        nome_depuratore = file_path.stem.replace(" ELABORATI", "").strip()
        assert nome_depuratore in impianto_to_comune, f"Impianto {nome_depuratore} non trovato nel mapping impianto_to_comune."
        
        df = pd.read_excel(file_path, engine="odf")
        df["Impianto"] = nome_depuratore
        
        dict_depuratori[nome_depuratore] = df
        all_dfs.append(df)
        
        print(f"✅ Caricato: {nome_depuratore} ({len(df)} righe)")

    df_totale = pd.concat(all_dfs, ignore_index=True)
    if save:
        df_totale.to_csv(folder_path / "JOINED" /"DF_JOINED_DEPURATORI.csv", index=False)
    print("\nDataset unificato creato con successo! Forma:", df_totale.shape)
else: 
    df_totale = pd.read_csv(folder_path / "JOINED" / "DF_JOINED_DEPURATORI.csv")

In [ ]:
df_totale['comune'] = df_totale['Impianto'].map(impianto_to_comune)  # mapping impianto to comune

In [ ]:
df_totale["ID_COMUNE"]= df_totale["comune"].map(json_comuni)
df_totale['ID_COMUNE'] = df_totale['ID_COMUNE'].astype(str).str.zfill(6)
df_totale[df_totale['ID_COMUNE'].isna()].comune.unique()

In [ ]:
vp_comuni = set(
    vodafone_presenze["comune"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .unique()
)

dep_comuni = set(
    df_totale["comune"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .unique()
)

print("Solo in Vodafone:", sorted(vp_comuni - dep_comuni))
print("Solo in depurazione:", sorted(dep_comuni - vp_comuni))

## A noi interessa considerare i comuni in DEPURAZIONE (tutti presenti, correttamente, in Vodafone) 

In [ ]:
vodafone_monthly_comune = vodafone_presenze.groupby(["mese", "ID_COMUNE", "userProfile"], as_index=False).agg({'value' : 'sum', 'comune' : 'first'})  # sommiamo i valori per mese, APT e userProfile

vodafone_monthly_comune['anno'] = vodafone_monthly_comune['mese'].dt.year
vodafone_monthly_comune['mese_num'] = vodafone_monthly_comune['mese'].dt.month

df_totale['Data'] = pd.to_datetime(df_totale['Data'])
df_totale['anno'] = df_totale['Data'].dt.year
df_totale['mese_num'] = df_totale['Data'].dt.month


df_presenze_prep_comunale = (
    vodafone_monthly_comune[vodafone_monthly_comune['userProfile'].isin(userprofile)]
    .groupby(['comune', 'anno', 'mese_num'], as_index=False)['value']
    .sum()
    .sort_values('mese_num')
)
df_presenze_prep_comunale

In [ ]:
df_totale = df_totale.sort_values('mese_num')

# Escludiamo i campioni flaggati come OUTLIER dal calcolo della media 
mask_no_outlier = df_totale['Outlier'].isna() if 'Outlier' in df_totale.columns else True

df_ae_monthly = (
    df_totale[mask_no_outlier]
    .dropna(subset=['AE'])
    .groupby(['comune', 'anno', 'mese_num'], as_index=False)['AE']
    .mean()
)
for aa in df_totale["comune"].unique():
    presenze_comune_aa = df_presenze_prep_comunale[df_presenze_prep_comunale['comune'] == aa]
    depurazione_aa = df_ae_monthly[df_ae_monthly['comune'] == aa].sort_values('mese_num')

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    
    for anno, dati in presenze_comune_aa.groupby('anno'):
        dati = dati.sort_values('mese_num')
        fig.add_trace(
            go.Scatter(x=dati['mese_num'], y=dati['value'],
                       mode='lines+markers', name=f'Presenze {anno}',
                       visible='legendonly'),   # <-- nascosta di default
            secondary_y=False,
        )

    for anno, dati in depurazione_aa.groupby('anno'):
        fig.add_trace(
            go.Scatter(
                x=dati['mese_num'],
                y=dati['AE'],
                mode='lines+markers',
                name=f'AE medio {anno}',
                line=dict(dash='dot'),
                visible='legendonly',  
            ), 
            secondary_y=True
        )

    # Media annuale Presenze - VISIBILE di default 
    mean_df_presenze = presenze_comune_aa.groupby('mese_num').agg({"value": "mean"})
    fig.add_trace(
        go.Scatter(
            x=mean_df_presenze.index,
            y=mean_df_presenze.value,
            mode='lines+markers',
            name='Presenze, media annuale',
            line=dict(dash='dot', color="orange"),
        )
    )

    # Media annuale AE - VISIBILE di default
    mean_df_ae = depurazione_aa.groupby('mese_num').agg({"AE": "mean"})
    fig.add_trace(
        go.Scatter(
            x=mean_df_ae.index,
            y=mean_df_ae.AE,
            mode='lines+markers',
            name='AE, media annuale',
            line=dict(dash='dot', color="purple", width=3),
        ),
        secondary_y=True,
    )

    fig.update_layout(
        title=f'<b>{aa}</b> - Presenze vs AE medio mensile',
        xaxis=dict(tickmode='array', tickvals=list(range(1, 13)),
                   ticktext=['Gen','Feb','Mar','Apr','Mag','Giu',
                             'Lug','Ago','Set','Ott','Nov','Dic']),
        template='plotly_white',
    )
    fig.update_yaxes(title_text='Presenze Totali', secondary_y=False)
    fig.update_yaxes(title_text='AE medio', secondary_y=True)
    fig.show()

In [ ]:
presenze_medie_comunale = df_presenze_prep_comunale.groupby(['comune', 'mese_num'], as_index=False)['value'].mean()
ae_medie_comunale = df_ae_monthly.groupby(['comune', 'mese_num'], as_index=False)['AE'].mean()

df_corr_comunale = pd.merge(
    presenze_medie_comunale,
    ae_medie_comunale,
    on=['comune', 'mese_num']
)

correlazioni_comunale = (
    df_corr_comunale.groupby('comune')[['value']]
    .corrwith(df_corr_comunale['AE'])
    .reset_index()
    .rename(columns={'value': 'Correlazione'})
).sort_values('Correlazione', ascending=False)

print('--- CORRELAZIONE AE vs PRESENZE PER COMUNE ---')
print(correlazioni_comunale)

In [ ]:
correlazioni_plot = correlazioni_comunale.sort_values('Correlazione', ascending=False)

fig_corr_matrix = px.imshow(
    correlazioni_plot[['Correlazione']].values,
    y=correlazioni_plot['comune'],
    x=['AE vs Presenze'],
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1,
    text_auto='.2f',
    aspect='auto',
)

fig_corr_matrix.update_layout(
    title='<b>Correlazione AE vs Presenze per Comune (dal più alto al più basso)</b>',
    height=max(400, 25 * len(correlazioni_plot)),
    template='plotly_white',
    coloraxis_colorbar=dict(title='Correlazione'),
)
fig_corr_matrix.update_xaxes(showticklabels=False)

fig_corr_matrix.show()